In [1]:
%pip install -q pymongo pandas numpy matplotlib ipywidgets


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import configparser
import re
import math
from datetime import datetime, timezone, time, timedelta

import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

from pymongo import MongoClient
from IPython.display import display, clear_output, HTML

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
# CONNECT TO MONGODB USING EXTERNAL database.conf

def connect_to_mongodb(config_file="database.conf"):
    config = configparser.ConfigParser()
    files_read = config.read(config_file)

    if not files_read:
        raise FileNotFoundError(
            f"Could not find {config_file}. "
            "Place database.conf in the same folder as this notebook."
        )

    if "mongodb" not in config:
        raise KeyError(
            "The [mongodb] section was not found in database.conf."
        )

    mongo = config["mongodb"]

    host = mongo.get("host", "localhost")
    port = mongo.getint("port", 27017)
    database_name = mongo.get("database", "usgs")
    collection_name = mongo.get("collection", "earthquakes")
    username = mongo.get("username", "").strip()
    password = mongo.get("password", "").strip()
    auth_source = mongo.get("authSource", "admin").strip() or "admin"

    client_args = {
        "host": host,
        "port": port,
        "serverSelectionTimeoutMS": 5000
    }

    if username and password:
        client_args.update({
            "username": username,
            "password": password,
            "authSource": auth_source
        })

    client = MongoClient(**client_args)
    client.admin.command("ping")

    db = client[database_name]
    collection = db[collection_name]

    return client, db, collection


try:
    client, db, collection = connect_to_mongodb("database.conf")
    print("MongoDB connection successful.")
    print("Database:", db.name)
    print("Collection:", collection.name)
    print("Earthquake records:", collection.count_documents({}))
except Exception as error:
    client = None
    db = None
    collection = None
    print("MongoDB connection failed:")
    print(error)

MongoDB connection successful.
Database: usgs
Collection: earthquakes
Earthquake records: 6701


In [4]:
# DATA TRANSFORMATION HELPERS

MAX_DASHBOARD_RECORDS = 5000

def start_date_to_milliseconds(date_value):
    if date_value is None:
        return None

    dt = datetime.combine(
        date_value,
        time.min,
        tzinfo=timezone.utc
    )

    return int(dt.timestamp() * 1000)


def end_date_to_milliseconds(date_value):
    if date_value is None:
        return None

    next_day = date_value + timedelta(days=1)

    dt = datetime.combine(
        next_day,
        time.min,
        tzinfo=timezone.utc
    )

    return int(dt.timestamp() * 1000) - 1


def documents_to_dataframe(documents):
    rows = []

    for document in documents:
        properties = document.get("properties", {}) or {}
        geometry = document.get("geometry", {}) or {}
        coordinates = geometry.get("coordinates", []) or []

        rows.append({
            "id": document.get("id"),
            "magnitude": properties.get("mag"),
            "place": properties.get("place"),
            "time_ms": properties.get("time"),
            "status": properties.get("status"),
            "longitude": coordinates[0] if len(coordinates) > 0 else None,
            "latitude": coordinates[1] if len(coordinates) > 1 else None,
            "depth": coordinates[2] if len(coordinates) > 2 else None
        })

    df = pd.DataFrame(rows)

    if df.empty:
        return df

    for column in ["magnitude", "longitude", "latitude", "depth"]:
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce"
        )

    df["time"] = pd.to_datetime(
        df["time_ms"],
        unit="ms",
        utc=True,
        errors="coerce"
    )

    df["place"] = df["place"].fillna("Unknown location")
    df["status"] = df["status"].fillna("Unknown")

    return df

print("Transformation helpers ready.")


Transformation helpers ready.


In [5]:
# QUERY FUNCTIONS
PAGE_SIZE = 10
CHART_RECORD_LIMIT = 5000


def build_query(
    minimum_magnitude,
    maximum_depth,
    start_date,
    end_date,
    place
):
    query = {}

    if minimum_magnitude is not None:
        query["properties.mag"] = {
            "$gte": float(minimum_magnitude)
        }

    if maximum_depth is not None:
        query["geometry.coordinates.2"] = {
            "$lte": float(maximum_depth)
        }

    time_filter = {}

    if start_date is not None:
        time_filter["$gte"] = start_date_to_milliseconds(start_date)

    if end_date is not None:
        time_filter["$lte"] = end_date_to_milliseconds(end_date)

    if time_filter:
        query["properties.time"] = time_filter

    if place.strip():
        query["properties.place"] = {
            "$regex": re.escape(place.strip()),
            "$options": "i"
        }

    return query


def get_chart_data(query):
    """
    Retrieve a limited number of matching records for the map
    and earthquakes-over-time visualization.
    """
    if collection is None:
        raise RuntimeError("MongoDB is not connected.")

    projection = {
        "id": 1,
        "properties.mag": 1,
        "properties.place": 1,
        "properties.time": 1,
        "properties.status": 1,
        "geometry.coordinates": 1
    }

    cursor = (
        collection
        .find(query, projection)
        .sort("properties.time", -1)
        .limit(CHART_RECORD_LIMIT)
    )

    return documents_to_dataframe(list(cursor))


def get_page_data(query, page_number):
    """
    Retrieve exactly one page of table results from MongoDB.
    Each page contains PAGE_SIZE (10) records.
    """
    if collection is None:
        raise RuntimeError("MongoDB is not connected.")

    total_matches = collection.count_documents(query)

    total_pages = max(
        1,
        math.ceil(total_matches / PAGE_SIZE)
    )

    page_number = max(
        1,
        min(page_number, total_pages)
    )

    skip_count = (
        page_number - 1
    ) * PAGE_SIZE

    projection = {
        "id": 1,
        "properties.mag": 1,
        "properties.place": 1,
        "properties.time": 1,
        "properties.status": 1,
        "geometry.coordinates": 1
    }

    cursor = (
        collection
        .find(query, projection)
        .sort("properties.time", -1)
        .skip(skip_count)
        .limit(PAGE_SIZE)
    )

    df = documents_to_dataframe(list(cursor))

    return (
        df,
        total_matches,
        total_pages,
        page_number
    )


print("Query and pagination functions ready.")

Query and pagination functions ready.


In [6]:
# DETERMINE AVAILABLE DATE RANGE

def get_date_bounds():
    if collection is None:
        return None, None

    earliest = list(
        collection.find(
            {"properties.time": {"$ne": None}},
            {"properties.time": 1}
        )
        .sort("properties.time", 1)
        .limit(1)
    )

    latest = list(
        collection.find(
            {"properties.time": {"$ne": None}},
            {"properties.time": 1}
        )
        .sort("properties.time", -1)
        .limit(1)
    )

    earliest_date = None
    latest_date = None

    if earliest:
        earliest_ms = earliest[0]["properties"]["time"]
        earliest_date = pd.to_datetime(
            earliest_ms,
            unit="ms",
            utc=True
        ).date()

    if latest:
        latest_ms = latest[0]["properties"]["time"]
        latest_date = pd.to_datetime(
            latest_ms,
            unit="ms",
            utc=True
        ).date()

    return earliest_date, latest_date


earliest_date, latest_date = get_date_bounds()

print("Earliest earthquake:", earliest_date)
print("Latest earthquake:", latest_date)

Earliest earthquake: 2025-03-16
Latest earthquake: 2025-04-15


In [7]:
# STORYBOARD-MATCHING CONTROLS

minimum_magnitude_widget = widgets.FloatText(
    value=0.0,
    description="Minimum Magnitude:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="230px")
)

maximum_depth_widget = widgets.FloatText(
    value=1000.0,
    description="Maximum Depth:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="220px")
)

place_widget = widgets.Text(
    value="",
    placeholder="Example: Alaska",
    description="Location:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="260px")
)

start_date_widget = widgets.DatePicker(
    description="Start Date:",
    value=earliest_date,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="220px")
)

end_date_widget = widgets.DatePicker(
    description="End Date:",
    value=latest_date,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="220px")
)

search_button = widgets.Button(
    description="Search",
    button_style="primary",
    icon="search",
    layout=widgets.Layout(width="120px")
)

# Pagination controls
previous_page_button = widgets.Button(
    description="Previous",
    icon="arrow-left",
    layout=widgets.Layout(width="120px")
)

next_page_button = widgets.Button(
    description="Next",
    icon="arrow-right",
    layout=widgets.Layout(width="120px")
)

page_number_widget = widgets.BoundedIntText(
    value=1,
    min=1,
    max=1,
    step=1,
    description="Page:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="140px")
)

page_status_widget = widgets.HTML(
    value="<b>Page 1 of 1</b>"
)

dashboard_output = widgets.Output()
results_output = widgets.Output()

# State used by pagination
current_query = {}
current_page = 1
current_total_matches = 0
current_total_pages = 1

print("Dashboard and pagination controls ready.")

Dashboard and pagination controls ready.


In [8]:
# DASHBOARD VISUALIZATION

def display_dashboard_charts(df, total_matches):
    """
    Display the two storyboard visualizations.
    The results table is rendered separately so page changes
    do not redraw the charts.
    """

    if df.empty:
        display(
            HTML(
                "<h3>No earthquake records matched the selected filters.</h3>"
            )
        )
        return

    display(
        HTML(
            f"<p><b>{total_matches:,}</b> earthquake records match the current search.</p>"
        )
    )

    # EARTHQUAKE MAP

    display(HTML("<h3>Earthquake Map</h3>"))

    location_data = df.dropna(
        subset=["longitude", "latitude"]
    ).copy()

    if not location_data.empty:
        point_sizes = (
            (
                location_data["magnitude"]
                .fillna(0)
                .clip(lower=0)
                + 1
            ) ** 2
        ) * 8

        fig, ax = plt.subplots(figsize=(10, 4.2))

        ax.scatter(
            location_data["longitude"],
            location_data["latitude"],
            s=point_sizes,
            alpha=0.45
        )

        ax.set_title("Earthquake Map")
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
        ax.set_xlim(-180, 180)
        ax.set_ylim(-90, 90)
        ax.grid(alpha=0.25)

        plt.tight_layout()
        plt.show()

    # EARTHQUAKES OVER TIME

    display(HTML("<h3>Earthquakes Over Time</h3>"))

    time_data = df.dropna(
        subset=["time"]
    ).copy()

    if not time_data.empty:
        time_data["date"] = time_data["time"].dt.floor("D")
        daily = time_data.groupby("date").size()

        fig, ax = plt.subplots(figsize=(10, 4.2))

        ax.plot(
            daily.index,
            daily.values,
            marker="o"
        )

        ax.set_title("Earthquakes Over Time")
        ax.set_xlabel("Date")
        ax.set_ylabel("Number of Earthquakes")
        ax.grid(alpha=0.25)

        fig.autofmt_xdate()
        plt.tight_layout()
        plt.show()


def display_results_page():
    """
    Display only the current 10-row results page.
    """
    global current_page
    global current_total_matches
    global current_total_pages

    with results_output:
        clear_output(wait=True)

        try:
            (
                page_df,
                current_total_matches,
                current_total_pages,
                current_page
            ) = get_page_data(
                current_query,
                current_page
            )

            page_number_widget.max = current_total_pages
            page_number_widget.value = current_page

            page_status_widget.value = (
                f"<b>Page {current_page} of {current_total_pages}</b>"
                f"&nbsp;&nbsp;|&nbsp;&nbsp;"
                f"{current_total_matches:,} total matching records"
            )

            previous_page_button.disabled = (
                current_page <= 1
            )

            next_page_button.disabled = (
                current_page >= current_total_pages
            )

            display(HTML("<h3>Search Results</h3>"))

            if page_df.empty:
                display(
                    HTML(
                        "<p>No records are available on this page.</p>"
                    )
                )
                return

            table = (
                page_df[
                    [
                        "magnitude",
                        "place",
                        "depth",
                        "time",
                        "status"
                    ]
                ]
                .copy()
            )

            table["time"] = table["time"].dt.strftime(
                "%Y-%m-%d %H:%M:%S UTC"
            )

            table = table.rename(
                columns={
                    "magnitude": "Magnitude",
                    "place": "Location",
                    "depth": "Depth",
                    "time": "Time",
                    "status": "Status"
                }
            )

            first_record = (
                (current_page - 1) * PAGE_SIZE
            ) + 1

            last_record = min(
                current_page * PAGE_SIZE,
                current_total_matches
            )

            display(
                HTML(
                    f"<p>Showing records "
                    f"<b>{first_record:,}-{last_record:,}</b> "
                    f"of <b>{current_total_matches:,}</b>.</p>"
                )
            )

            display(
                table.reset_index(drop=True)
            )

        except Exception as error:
            print("Results table error:")
            print(error)

In [9]:
# SEARCH AND PAGINATION ACTIONS

def refresh_dashboard(button=None):
    global current_query
    global current_page
    global current_total_matches
    global current_total_pages

    with dashboard_output:
        clear_output(wait=True)

        try:
            if collection is None:
                raise RuntimeError(
                    "MongoDB connection is not available."
                )

            if (
                start_date_widget.value is not None
                and end_date_widget.value is not None
                and start_date_widget.value > end_date_widget.value
            ):
                raise ValueError(
                    "Start Date cannot be later than End Date."
                )

            current_query = build_query(
                minimum_magnitude_widget.value,
                maximum_depth_widget.value,
                start_date_widget.value,
                end_date_widget.value,
                place_widget.value
            )

            # Every new search begins on page 1.
            current_page = 1

            current_total_matches = (
                collection.count_documents(
                    current_query
                )
            )

            current_total_pages = max(
                1,
                math.ceil(
                    current_total_matches / PAGE_SIZE
                )
            )

            chart_df = get_chart_data(
                current_query
            )

            display_dashboard_charts(
                chart_df,
                current_total_matches
            )

        except Exception as error:
            print("Dashboard error:")
            print(error)

    # Render page 1 at the bottom after the charts.
    display_results_page()


def go_to_previous_page(button=None):
    global current_page

    if current_page > 1:
        current_page -= 1
        display_results_page()


def go_to_next_page(button=None):
    global current_page

    if current_page < current_total_pages:
        current_page += 1
        display_results_page()


def jump_to_page(change):
    global current_page

    if change["name"] != "value":
        return

    requested_page = int(change["new"])

    if requested_page != current_page:
        current_page = requested_page
        display_results_page()


search_button.on_click(
    refresh_dashboard
)

previous_page_button.on_click(
    go_to_previous_page
)

next_page_button.on_click(
    go_to_next_page
)

page_number_widget.observe(
    jump_to_page,
    names="value"
)

print("Search and pagination actions connected.")

Search and pagination actions connected.


In [10]:
# STORYBOARD-MATCHING LAYOUT

dashboard_title = widgets.HTML(
    value=(
        "<h2 style='margin-bottom:4px;'>USGS Earthquake Dashboard</h2>"
        "<p style='margin-top:0;'>"
        "Search earthquake data by magnitude, depth, date range, and location."
        "</p>"
    )
)

filter_row_1 = widgets.HBox(
    [
        minimum_magnitude_widget,
        maximum_depth_widget,
        place_widget
    ]
)

filter_row_2 = widgets.HBox(
    [
        start_date_widget,
        end_date_widget,
        search_button
    ]
)

dashboard_controls = widgets.VBox(
    [
        dashboard_title,
        filter_row_1,
        filter_row_2
    ]
)

pagination_controls = widgets.HBox(
    [
        previous_page_button,
        page_number_widget,
        page_status_widget,
        next_page_button
    ]
)

display(dashboard_controls)

# Map and time chart
display(dashboard_output)

# Paginated table at the bottom
display(
    widgets.HTML(
        value="<hr><h2>Earthquake Results</h2>"
    )
)

display(pagination_controls)
display(results_output)

# Load initial dashboard and first page.
refresh_dashboard()

Output()

HTML(value='<hr><h2>Earthquake Results</h2>')

Output()